# 03 - Train the segmentation network on synthetic tumors, then test on BraTS

**Press Run All.** Needs: Accelerator = **GPU (T4 or P100)**, Internet = On, the Kaggle Secret `GITHUB_TOKEN`, and these three datasets added as Input (the ones made from `hcp_lean.zip`, `tumor_bank.zip`, `brats_test.zip`).

What it does: installs MONAI -> downloads our code -> trains a 3D U-Net on fake tumors that are made on the fly (the healthy scans + the 1000-tumor bank) -> plots the loss -> tests on the 31 BraTS cases (DSC, HD95, AUPRC, SEN) -> zips the model.

**First run it as a PILOT** (`ITERS = 300`, about 10-20 minutes): it checks that everything works and tells you the seconds per iteration. Then set `ITERS` to a bigger number (the cell below tells you how long it will take) and run again.

**Recommended way to run (so nothing is lost when Kaggle ends the session):** top right **Save Version -> Save & Run All (Commit)**. It runs in the background; afterwards open the version's **Output** tab to download the zip. The notebook also shows `pictures.png` (worst / middle / best BraTS cases: green = true tumor, red = predicted).

## 0. Settings

In [ ]:
RUN_NAME = "c2_aug"   # folder name for this run (use a new name for every new run)
ITERS = 10000             # 300 = pilot. Full run: see the time estimate printed after the pilot
CLASSES = 6             # 6 = LF-SynthSeg with tissue labels (C2). 2 = tumor-only (C1)
BATCH = 2
WORKERS = 3             # CPU workers that cut patches out of the healthy scans
MAX_MINUTES = None      # e.g. 600 = stop after 10 hours (it saves a checkpoint; run again to continue)
RUN_EVAL = True         # test on the 31 BraTS cases at the end

## 1. Check the GPU, install MONAI, get our code (needs the `GITHUB_TOKEN` secret)

In [ ]:
import os, subprocess, sys, glob
def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd[:80]}")
    return r.returncode

import torch
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2 (or P100), then Run All again"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)
sh(f"{sys.executable} -m pip install -q 'monai>=1.3' nibabel scipy scikit-learn matplotlib pyyaml", check=True)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO = "/kaggle/working/repo"
sh(f"cd /kaggle/working && rm -rf repo && git clone -q https://{token}@github.com/PiyushGarg0146/lf-synthseg-btp.git repo", check=True)
sh(f"cd {REPO} && git log --oneline | head -3", check=True)

## 2. Find the data (the three attached datasets)

In [ ]:
from pathlib import Path
inp = Path("/kaggle/input")
labels = sorted(inp.rglob("labels.nii*"))
HCP = str(labels[0].parents[1]) if labels else None
npz = sorted(inp.rglob("tumor_*.npz"))
BANK = str(npz[0].parent) if npz else None
tum = sorted(inp.rglob("tumor.nii*"))
BRATS = str(tum[0].parents[1]) if tum else None
print("healthy scans:", len(labels), "->", HCP)
print("tumor bank files:", len(npz), "->", BANK)
print("BraTS test cases:", len(tum), "->", BRATS)
assert len(labels) == 20 and len(npz) >= 1000 and len(tum) == 31, "a dataset is missing: add all three as Input"

## 3. Train (prints the loss and seconds per iteration)

In [ ]:
OUT = f"/kaggle/working/runs/{RUN_NAME}"
cmd = (f"cd {REPO} && {sys.executable} -u train/train.py --hcp {HCP} --bank {BANK} --out {OUT} --classes {CLASSES} "
       f"--iters {ITERS} --batch {BATCH} --workers {WORKERS}" + (f" --max-minutes {MAX_MINUTES}" if MAX_MINUTES else ""))
p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
last = ""
for line in p.stdout:
    print(line, end=""); last = line
p.wait()
assert p.returncode == 0, "training failed - see the messages above"

In [ ]:
import csv, matplotlib.pyplot as plt
rows = list(csv.DictReader(open(f"{OUT}/log.csv")))
it = [int(r["iter"]) for r in rows]; loss = [float(r["loss"]) for r in rows]
plt.figure(figsize=(7, 3.5)); plt.plot(it, loss); plt.xlabel("iteration"); plt.ylabel("loss (Dice + CE)"); plt.grid(alpha=.3); plt.show()
sec = float(rows[-1]["seconds"]) / int(rows[-1]["iter"])
print(f"about {sec:.2f} seconds per iteration -> 10,000 iterations = {sec * 10000 / 3600:.1f} h, 20,000 = {sec * 20000 / 3600:.1f} h")

## 4. Test on the 31 BraTS cases (whole tumor)

In [ ]:
if RUN_EVAL and os.path.exists(f"{OUT}/final.pt"):
    cmd = f"cd {REPO} && {sys.executable} -u eval/run_eval.py --ckpt {OUT}/final.pt --brats {BRATS} --out {OUT}/eval_brats2020"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    assert p.returncode == 0, "evaluation failed - see the messages above"
else:
    print("skipped (no final.pt yet, or RUN_EVAL = False)")

In [ ]:
from IPython.display import Image, display
pic = f"{OUT}/eval_brats2020/pictures.png"
if os.path.exists(pic):
    display(Image(pic))


## 5. Package results for download (the model, the log and the evaluation; not the big checkpoint)

In [ ]:
sh(f"cd /kaggle/working/runs && rm -f /kaggle/working/{RUN_NAME}.zip && zip -qr /kaggle/working/{RUN_NAME}.zip {RUN_NAME} -x '*/latest.pt' && ls -lh /kaggle/working/{RUN_NAME}.zip", check=True)
from IPython.display import FileLink
FileLink(f"/kaggle/working/{RUN_NAME}.zip")